# Minesweeper LLM Competition - Custom GRPO Training

## Goal
Finetune an LLM with LoRA using GRPO to play Minesweeper by:
- **Input**: JSON game state (board configuration)
- **Output**: JSON action (reveal or flag a cell)

Teams will compete to train the best Minesweeper-playing LLM!

## Training Approach
- **Model**: GPT-OSS 20B with LoRA or other models in the /root/.cache/huggingface/hub directory [**Any model other than /root/.cache/huggingface/hub will lead to disqualification**]
- **Method**: GRPO (Group Relative Policy Optimization), SFT or any RL-policies (not just strict to use GRPO)
- **Framework**: Unsloth (2-6x faster, 70% less VRAM)
- **Hardware**: AMD GPU (ROCm)

# Load Model with Unsloth

Load GPT-OSS 20B with LoRA configuration:

In [ ]:
import os
os.environ["HF_HUB_CACHE"] ="/root/.cache/huggingface"

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 6000  # Max context length
lora_rank = 16        # LoRA rank (higher = smarter but slower; 4 is too low for reasoning tasks)

# Try loading with explicit torch_dtype
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/root/.cache/huggingface/models--Unsloth--Llama-3.1-8B-Instruct/snapshots/4699cc75b550f9c6f3173fb80f4703b62d946aa5",
    load_in_4bit = False,
    max_seq_length = max_seq_length,
    torch_dtype = torch.bfloat16,
)

# Force model to cuda explicitly
print(f"Model device: {model.device}")
print("Model loaded successfully!")

# Add LoRA Adapters

Add LoRA layers for efficient finetuning:

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Minesweeper Game Implementation

Custom Minesweeper environment supporting:
- Customizable board size and mine count
- Actions: reveal or flag cells
- Win: reveal all safe cells
- Lose: reveal a mine

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Set
import random

@dataclass
class MinesweeperGame:
    rows: int
    cols: int
    num_mines: int
    seed: Optional[int] = None
    _rng: random.Random = field(init=False, repr=False)
    _board: List[List[int]] = field(init=False, repr=False)  # -1 = mine, 0-8 = count
    _revealed: Set[Tuple[int, int]] = field(init=False, repr=False, default_factory=set)
    _flagged: Set[Tuple[int, int]] = field(init=False, repr=False, default_factory=set)
    _state: str = field(default="ongoing", init=False, repr=False)

    def __post_init__(self):
        if self.num_mines >= self.rows * self.cols:
            raise ValueError("Too many mines for board size")
        self._rng = random.Random(self.seed)
        self._board = [[0 for _ in range(self.cols)] for _ in range(self.rows)]
        self._place_mines()
        self._calculate_numbers()
        
        # --- NEW CODE START ---
        # Reveal a random safe cell so the AI starts with some information
        # (effectively simulating the first move having been played)
        safe_cells = [(r, c) for r in range(self.rows) for c in range(self.cols) if self._board[r][c] != -1]
        
        if safe_cells:
            start_r, start_c = self._rng.choice(safe_cells)
            self._reveal_cell(start_r, start_c)
        # --- NEW CODE END ---
    def _place_mines(self):
        """Place mines randomly on the board"""
        positions = [(r, c) for r in range(self.rows) for c in range(self.cols)]
        mine_positions = self._rng.sample(positions, self.num_mines)
        for r, c in mine_positions:
            self._board[r][c] = -1

    def _calculate_numbers(self):
        """Calculate numbers for each cell based on adjacent mines"""
        for r in range(self.rows):
            for c in range(self.cols):
                if self._board[r][c] == -1:
                    continue
                count = 0
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0:
                            continue
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < self.rows and 0 <= nc < self.cols:
                            if self._board[nr][nc] == -1:
                                count += 1
                self._board[r][c] = count

    def _reveal_cell(self, row: int, col: int) -> bool:
        """Reveal a cell. Returns True if valid move, False if invalid.
        Uses iterative flood-fill to avoid recursion limit on large boards.
        (Issue #11: was recursive; Issue typo: fixed 'bself' -> 'self')
        """
        if not (0 <= row < self.rows and 0 <= col < self.cols):
            return False
        if (row, col) in self._revealed or (row, col) in self._flagged:
            return False

        stack = [(row, col)]
        while stack:
            r, c = stack.pop()
            if (r, c) in self._revealed:
                continue

            self._revealed.add((r, c))

            # Hit a mine!
            if self._board[r][c] == -1:
                self._state = "failed"
                return True

            # Auto-reveal neighbors if cell is 0
            if self._board[r][c] == 0:
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0:
                            continue
                        nr, nc = r + dr, c + dc
                        if (0 <= nr < self.rows and 0 <= nc < self.cols
                                and (nr, nc) not in self._revealed
                                and (nr, nc) not in self._flagged):
                            stack.append((nr, nc))

        return True

    def _flag_cell(self, row: int, col: int) -> bool:
        """Flag/unflag a cell. Returns True if valid, False if invalid"""
        if not (0 <= row < self.rows and 0 <= col < self.cols):
            return False
        if (row, col) in self._revealed:
            return False
        
        if (row, col) in self._flagged:
            self._flagged.remove((row, col))
        else:
            self._flagged.add((row, col))
        return True

    def do_action(self, action: dict) -> str:
        """Execute an action and return a status string.

        Returns one of:
          'ok'               - valid move executed
          'mine'             - revealed a mine (game over)
          'win'              - game won after this move
          'invalid_format'   - bad action dict / missing keys / bad types
          'out_of_bounds'    - coordinates outside the board
          'already_revealed' - cell was already revealed
          'flagged_cell'     - tried to reveal a flagged cell
          'invalid_flag'     - tried to flag a revealed cell
          'game_over'        - game was already over before this call

        (Issue #13: previously set state='failed' for ALL invalid moves,
         conflating formatting errors with hitting a mine.)
        """
        if self._state != "ongoing":
            return "game_over"

        if not isinstance(action, dict):
            self._state = "failed"
            return "invalid_format"

        action_type = action.get("type")
        row = action.get("row")
        col = action.get("col")

        if action_type not in ["reveal", "flag"] or row is None or col is None:
            self._state = "failed"
            return "invalid_format"

        try:
            row, col = int(row), int(col)
        except (ValueError, TypeError):
            self._state = "failed"
            return "invalid_format"

        if not (0 <= row < self.rows and 0 <= col < self.cols):
            self._state = "failed"
            return "out_of_bounds"

        if action_type == "reveal":
            if (row, col) in self._revealed:
                self._state = "failed"
                return "already_revealed"
            if (row, col) in self._flagged:
                self._state = "failed"
                return "flagged_cell"
            valid = self._reveal_cell(row, col)
        else:
            if (row, col) in self._revealed:
                self._state = "failed"
                return "invalid_flag"
            valid = self._flag_cell(row, col)

        if not valid:
            self._state = "failed"
            return "invalid_format"

        self._check_win()

        if self._state == "failed":
            return "mine"
        if self._state == "success":
            return "win"
        return "ok"

    def _check_win(self):
        """Check if player has won"""
        total_cells = self.rows * self.cols
        safe_cells = total_cells - self.num_mines
        if len(self._revealed) == safe_cells:
            self._state = "success"

    def get_visible_board(self) -> List[List[str]]:
        """Get board state as player sees it"""
        visible = []
        for r in range(self.rows):
            row = []
            for c in range(self.cols):
                if (r, c) in self._flagged:
                    row.append('F')
                elif (r, c) in self._revealed:
                    val = self._board[r][c]
                    row.append('*' if val == -1 else str(val))
                else:
                    row.append('.')
            visible.append(row)
        return visible

    def state(self) -> str:
        return self._state

    def pretty_print(self) -> str:
        """Pretty print the board"""
        visible = self.get_visible_board()
        lines = []
        
        # Header
        header = "   " + " ".join(f"{i:2d}" for i in range(self.cols))
        lines.append(header)
        lines.append("  " + "─" * (self.cols * 3 + 1))
        
        # Board
        for r, row in enumerate(visible):
            line = f"{r:2d}│ " + "  ".join(row)
            lines.append(line)
        
        return "\n".join(lines)

# Test the Game

In [ ]:
# Create test game
game = MinesweeperGame(rows=6, cols=6, num_mines=5)
print(game.pretty_print())
print(f"State: {game.state()}")

# Test action
game.do_action({"type": "reveal", "row": 0, "col": 0})
print("\nAfter revealing (0,0):")
print(game.pretty_print())
print(f"State: {game.state()}")

# JSON Input/Output Format

## Input Format (Game State)
```json
{
  "board": [
    ["1", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."]
  ],
  "rows": 6,
  "cols": 6,
  "mines": 5,
  "flags_placed": 0,
  "cells_revealed": 0
}
```

## Output Format (Action)
```json
{"type": "reveal", "row": 2, "col": 3}
```
or
```json
{"type": "flag", "row": 1, "col": 4}
```

In [ ]:
import json

'''
Important Hints:

1. Prompt is crucial - make sure your LLM is not verbose and do not write/output reasoning, instead the verbose must be hidden or abstracted and
    output must be JSON object - the verbosity in our experiment led to running out of max tokens set and
    thus JSON parsing failure - i.e. Disqualification:
    {{"type": "reveal", "row": <row_index>, "col": <col_index>}}
    or
    {{"type": "flag", "row": <row_index>, "col": <col_index>}}

2. Make sure your model learns generic N*M game board shapes and # number of mines

3. Do not flag the cell which is already flagged - game will go in recursion and you will have heavy penalty

4. Do not flag the cell which is already revealed - game will go in recursion and you will have heavy penalty
'''

def format_state_for_llm(game: MinesweeperGame, hide_hints= False) -> str:
    """Convert game state to JSON prompt for LLM"""
    state = {
        "board": game.get_visible_board(),
        "rows": game.rows,
        "cols": game.cols,
        "mines": game.num_mines,
        "flags_placed": len(game._flagged),
        "cells_revealed": len(game._revealed),
    }

    prompt = f"""
You are an expert Minesweeper solver. Your goal is to maximize your score by playing optimally.

## Scoring System
- **Win Game**: +100 points (Flag all mines + Reveal all safe cells)
- **Flag Mine**: +15 points
- **Reveal Safe Cell (Logical)**: +15 points
- **Reveal Safe Cell (Guess)**: +10 points
- **Flag Incorrectly**: -10 points
- **Reveal Mine**: -25 points (Game Over)
- **Redundant Move (Reveal/Flag again)**: -12 / -8 points
- **Invalid Move (Out of bounds/JSON)**: -15 / -10 points
- **Excessive Flags**: -10 points

## Strategy
1. **Prioritize Safety**: Hitting a mine ends the round with a heavy penalty (-25).
2. **Maximize Points**: 
   - logically deduced safe cells give +15.
   - Flagging confirmed mines gives +15.
   - Guessing gives +10, but carries risk of hitting a mine.
3. **Avoid Penalties**: Never repeat a move. Ensure coordinates are valid. Do not flag more than remaining mines.

## Board State
{json.dumps(state)}

## Output Format
Respond with ONLY a JSON object (no other text outside the JSON):
```json
{{"type": "reveal", "row": <r>, "col": <c>}}
```
Or to flag a mine:
```json
{{"type": "flag", "row": <r>, "col": <c>}}
```

## Critical Rules
- DO NOT CLICK THE SAME GRID AS PREVIOUSLY CLICKED 
- DO NOT WRITE ANYTHING OTHER THAN DESCRIBED JSON. NO TEXT BEFORE OR AFTER, NO REASONING
- NEVER reveal a cell you've deduced is a mine.
- NEVER flag a cell you've deduced is safe.
- Prefer actions with CERTAINTY over guesses.
- When guessing, minimize risk — choose cells with the fewest adjacent unrevealed mines.
"""
    
    return prompt

def parse_llm_action(response: str) -> dict:
    """Extract JSON action from LLM response.
    
    Finds all JSON-like objects and returns the LAST one matching the
    expected schema.  LLMs typically reason through options and place
    their final answer at the end, so taking the last valid match is
    more robust than taking the first.
    """
    import re
    best = None
    for match in re.finditer(r'\{[^{}]*\}', response):
        try:
            action = json.loads(match.group())
            if ("type" in action and "row" in action and "col" in action
                    and action["type"] in ["reveal", "flag"]):
                best = action
        except json.JSONDecodeError:
            continue
    return best

# Test formatting
game = MinesweeperGame(rows=6, cols=6, num_mines=5)
prompt = format_state_for_llm(game)
print(prompt)

# Test Model Before Training

See how the base model performs without finetuning:

In [ ]:
from transformers import TextStreamer

game = MinesweeperGame(rows=6, cols=6, num_mines=5, seed=42)
prompt = format_state_for_llm(game)

# Removed reasoning_effort="low" — GRPOTrainer does NOT pass it
# during training, so using it only at eval creates a train/eval mismatch.
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
)

print("=== Base Model Response ===")
output = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 128,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

# GRPO Reward Functions

Define reward functions to guide the model's learning:

In [ ]:
"""
GRPO Reward Functions for Minesweeper LLM Training.

Uses the multi-axis reward system (Safety, Info Gain,
Efficiency, Flag Quality) combined with the 12 scoring criteria.

Designed to be passed directly to the GRPO trainer as reward functions.
Each function takes (completions, **kwargs) and returns a list of float scores.
"""

import re
import json
import math
import numpy as np
from collections import deque

# MinesweeperGame is already defined in an earlier cell — no import needed.


# ═══════════════════════════════════════════════════════════════════════════
# Constants — matching minesweeper.py reward values
# ═══════════════════════════════════════════════════════════════════════════

MINE = -1
CLOSED = -2
FLAG = -3


# ═══════════════════════════════════════════════════════════════════════════
# LLM Response Parsing
# ═══════════════════════════════════════════════════════════════════════════

def parse_llm_action(response: str) -> dict | None:
    """
    Parse an LLM response string to extract a Minesweeper action.

    Looks for JSON matching:
        {"reasoning": "...", "type": "reveal"|"flag", "row": int, "col": int}

    Handles common LLM formatting issues:
        - JSON inside markdown code blocks (```json ... ```)
        - Extra whitespace, trailing commas
        - Missing reasoning field

    Returns None if no valid action can be parsed.
    """
    if not response or not isinstance(response, str):
        return None

    # Strategy 1: Find JSON in markdown code block
    code_block = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response, re.DOTALL)
    if code_block:
        try:
            data = json.loads(code_block.group(1))
            return _validate_action(data)
        except json.JSONDecodeError:
            pass

    # Strategy 2: Find ANY JSON object in the response
    # Use greedy matching from last { to handle nested objects
    json_matches = re.findall(r'\{[^{}]*\}', response)
    for match in reversed(json_matches):  # try last match first (likely the action)
        try:
            data = json.loads(match)
            result = _validate_action(data)
            if result:
                return result
        except json.JSONDecodeError:
            continue

    # Strategy 3: Try to find JSON with nested braces (reasoning may contain them)
    brace_depth = 0
    start_idx = None
    for i, ch in enumerate(response):
        if ch == '{':
            if brace_depth == 0:
                start_idx = i
            brace_depth += 1
        elif ch == '}':
            brace_depth -= 1
            if brace_depth == 0 and start_idx is not None:
                try:
                    data = json.loads(response[start_idx:i+1])
                    result = _validate_action(data)
                    if result:
                        return result
                except json.JSONDecodeError:
                    pass
                start_idx = None

    return None


def _validate_action(data: dict) -> dict | None:
    """Validate that parsed JSON has the required action fields."""
    if not isinstance(data, dict):
        return None

    # Must have type + row + col
    action_type = data.get("type")
    row = data.get("row")
    col = data.get("col")

    if action_type not in ("reveal", "flag"):
        return None
    if not isinstance(row, (int, float)) or not isinstance(col, (int, float)):
        return None

    return {
        "reasoning": data.get("reasoning", ""),
        "type": action_type,
        "row": int(row),
        "col": int(col),
    }


# ═══════════════════════════════════════════════════════════════════════════
# Multi-Axis Reward Helpers (ported from minesweeper.py)
# ═══════════════════════════════════════════════════════════════════════════

def _sigmoid(x):
    """Sigmoid function for capping info-gain reward (PDF §5.3)."""
    return 1.0 / (1.0 + math.exp(-max(min(x, 500), -500)))


def _calculate_3bv(mine_board, rows, cols):
    """
    Calculate Bechtel's Board Value (3BV) — minimum clicks to solve
    without flags (PDF §6.1).

    Works on the hidden mine board (list of lists) from MinesweeperGame._board.
    """
    visited = [[False] * cols for _ in range(rows)]
    bv = 0

    def _count_mines(r, c):
        count = 0
        for dr in range(-1, 2):
            for dc in range(-1, 2):
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    if mine_board[nr][nc] == MINE:
                        count += 1
        return count

    # Step 1: Flood-fill zero regions
    for r in range(rows):
        for c in range(cols):
            if visited[r][c] or mine_board[r][c] == MINE:
                continue
            if _count_mines(r, c) == 0:
                bv += 1
                queue = deque([(r, c)])
                visited[r][c] = True
                while queue:
                    cr, cc = queue.popleft()
                    for dr in range(-1, 2):
                        for dc in range(-1, 2):
                            nr, nc = cr + dr, cc + dc
                            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr][nc]:
                                if mine_board[nr][nc] != MINE:
                                    visited[nr][nc] = True
                                    if _count_mines(nr, nc) == 0:
                                        queue.append((nr, nc))

    # Step 2: Count non-zero safe cells not yet visited
    for r in range(rows):
        for c in range(cols):
            if not visited[r][c] and mine_board[r][c] != MINE:
                bv += 1

    return max(bv, 1)


def _is_cell_deterministically_safe(visible_board, rows, cols, r, c):
    """
    Check if cell (r, c) is provably safe from the visible board alone (PDF §4.2).

    A cell is deterministically safe if ANY adjacent revealed number has all
    its mines already accounted for by flags.
    """
    for dr in range(-1, 2):
        for dc in range(-1, 2):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                cell_val = visible_board[nr][nc]
                if not cell_val.isdigit() or int(cell_val) == 0:
                    continue
                num = int(cell_val)

                # Count flags and hidden cells around this number
                flags = 0
                hidden_cells = []
                for ddr in range(-1, 2):
                    for ddc in range(-1, 2):
                        nnr, nnc = nr + ddr, nc + ddc
                        if 0 <= nnr < rows and 0 <= nnc < cols:
                            v = visible_board[nnr][nnc]
                            if v == "F":
                                flags += 1
                            elif v == ".":
                                hidden_cells.append((nnr, nnc))

                if num == flags and (r, c) in hidden_cells:
                    return True
    return False


def _count_unrevealed(visible_board, rows, cols):
    """Count hidden + flagged cells (i.e. cells not yet solved)."""
    count = 0
    for r in range(rows):
        for c in range(cols):
            if visible_board[r][c] in (".", "F"):
                count += 1
    return count


def _count_flags(visible_board, rows, cols):
    """Count flagged cells."""
    count = 0
    for r in range(rows):
        for c in range(cols):
            if visible_board[r][c] == "F":
                count += 1
    return count


# ═══════════════════════════════════════════════════════════════════════════
# Reward Function 1: Valid JSON Format
# ═══════════════════════════════════════════════════════════════════════════


def valid_json_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        action = parse_llm_action(response)
        # Small penalty for invalid, small bonus for valid to encourage structure
        # but not dominate the gameplay rewards.
        scores.append(0.2 if action is not None else -0.5)
    return scores

def gameplay_scores(completions, **kwargs):
    scores = []
    seeds = kwargs.get("seed", [])
    move_histories = kwargs.get("move_history", [])
    
    for idx, completion in enumerate(completions):
        response = completion[0]["content"]
        action = parse_llm_action(response)
        if action is None:
            scores.append(-2.0) # Was -50.0
            continue
            
        if idx >= len(seeds) or idx >= len(move_histories):
            scores.append(0.0)
            continue
            
        seed = seeds[idx]
        move_history_raw = move_histories[idx]
        if isinstance(move_history_raw, str):
             move_history = json.loads(move_history_raw)
        else:
             move_history = move_history_raw
             
        # Check for repetition of the immediate last move
        if len(move_history) > 0:
            last_move = move_history[-1]
            if last_move.get("row") == action["row"] and last_move.get("col") == action["col"]:
                scores.append(-5.0) # Was -100.0. Penalize repetition but don't kill the gradient.
                continue
             
        # Parse board size from dataset or default to 6x6
        rows, cols, num_mines = 6, 6, 5
        if "board_size" in kwargs:
             bs_list = kwargs["board_size"]
             # GRPO passes a list of values for the batch
             if idx < len(bs_list):
                 bs_str = bs_list[idx]
                 if "x" in bs_str:
                     r_str, c_str = bs_str.split("x")
                     rows, cols = int(r_str), int(c_str)
                     # Recalculate mines based on 20% density as per generation logic
                     num_mines = int(rows * cols * 0.20)
        
        game = MinesweeperGame(rows=rows, cols=cols, num_mines=num_mines, seed=seed)
        for prev_action in move_history:
            game.do_action(prev_action)
            
        row, col = action["row"], action["col"]
        action_type = action["type"]
        
        pre_state = game.state()
        result = game.do_action(action)
        
        score = 0.0
        # Scaled rewards (-1.0 to 1.0 range approximate)
        if result == 'win': score += 5.0         # Strong signal for winning
        elif result == 'mine': score += -0.5     # Negative but discoverable
        elif result == 'invalid_format': score += -0.5
        elif result == 'out_of_bounds': score += -2
        elif result == 'already_revealed': score += -3.0
        elif result == 'flagged_cell': score += -0.5
        elif result == 'invalid_flag': score += -1
        elif result == 'ok':
             if action_type == 'reveal': score += 0.5 # Progress
             elif action_type == 'flag': score += 0.25 # Caution
        
        scores.append(score)
    return scores

def combined_reward(completions, **kwargs):
    json_scores = valid_json_reward(completions, **kwargs)
    game_scores = gameplay_scores(completions, **kwargs)
    combined = []
    for j, g in zip(json_scores, game_scores):
        combined.append(j + g*1.5)
    return combined

def log_completions(completions, **kwargs):
    """
    Reward function that logs the first completion for debugging purposes.
    Returns 0.0 reward so it doesn't affect training.
    """
    if completions and len(completions) > 0:
        # Print the first completion content to stdout
        content = completions[0][0]["content"]
        # Basic cleanup to print readable debug output
        print(f"\n[DEBUG MODEL OUTPUT (len={len(content)})]:\n{content[:500]}...\n[END DEBUG]\n")
    return [0.0] * len(completions)


# ═══════════════════════════════════════════════════════════════════════════
# Self-test
# ═══════════════════════════════════════════════════════════════════════════

print("Reward functions defined successfully!")
print("Available: valid_json_reward, gameplay_scores, reasoning_quality_reward, combined_reward")

# Create Training Dataset

Generate diverse game states for training:

from datasets import Dataset
from datasets import concatenate_datasets

def generate_game_states(num_samples=1000, rows=6, cols=6, num_mines=5,
                         rng_seed=42):
    """
    Generate EXACTLY num_samples diverse Minesweeper game states.
    
    Mix of:
    - Fresh games (20-30%)
    - Mid-game states (70-80%)
    
    IMPORTANTLY: Stores seed + move_history (as JSON string) so reward
    function can reconstruct the EXACT game state!
    
    Keeps generating until we have exactly num_samples valid ongoing games.
    
    Args:
        rng_seed: Seed for numpy/random RNG for reproducibility.
    """
    # Seed RNG for reproducibility across runs
    np.random.seed(rng_seed)
    random.seed(rng_seed)

    dataset_items = []
    attempts = 0
    max_attempts = num_samples * 3  # Safety limit
    
    while len(dataset_items) < num_samples and attempts < max_attempts:
        attempts += 1
        seed = np.random.randint(100000)
        game = MinesweeperGame(rows=rows, cols=cols, num_mines=num_mines, seed=seed)
        
        # Make 0-5 random moves (0 = fresh game, 1-5 = mid-game)
        num_moves = np.random.randint(0, 6)
        move_history = []
        
        for _ in range(num_moves):
            board = game.get_visible_board()
            unrevealed = []
            for r in range(rows):
                for c in range(cols):
                    if board[r][c] == '.':
                        unrevealed.append((r, c))
            
            if unrevealed and game.state() == "ongoing":
                r, c = random.choice(unrevealed)
                action = {"type": "reveal", "row": r, "col": c}
                game.do_action(action)
                move_history.append(action)
            else:
                break
        
        # Only add ongoing games (skip failed/completed games)
        if game.state() == "ongoing":
            prompt_text = format_state_for_llm(game)
            dataset_items.append({
                "prompt": [{"role": "user", "content": prompt_text}],
                "seed": seed,  # Store seed to reconstruct game
                # IMPORTANT: Serialize as JSON string to avoid HF Dataset
                # schema inference mangling list-of-dicts into dict-of-lists
                "move_history": json.dumps(move_history),
            })
    
    return Dataset.from_list(dataset_items)

# Generate training dataset
print("Generating training dataset...")

ds1 = generate_game_states(num_samples=1000, rows=6, cols=6, num_mines = 5)
ds2 = generate_game_states(num_samples=1000, rows=10, cols=10, num_mines = 20)
ds3 = generate_game_states(num_samples=1000, rows=15, cols=15, num_mines = 45)
ds4 = generate_game_states(num_samples=1000, rows=20, cols=20, num_mines=80)
ds5 = generate_game_states(num_samples=1000, rows=25, cols=25, num_mines=125)

dataset = concatenate_datasets([ds1, ds2, ds3, ds4, ds5])
dataset = dataset.shuffle(seed=42)
print(f"Created EXACTLY {len(dataset)} training examples (all ongoing games)")

# Count fresh vs mid-game
fresh_count = sum(1 for item in dataset if item["move_history"] == "[]")
print(f"  Fresh games: {fresh_count} ({fresh_count/len(dataset)*100:.1f}%)")
print(f"  Mid-game states: {len(dataset) - fresh_count} ({(len(dataset)-fresh_count)/len(dataset)*100:.1f}%)")

# Show example
print("\nExample training prompt:")
print(dataset[0]["prompt"][0]["content"])
print(f"Seed: {dataset[0]['seed']}, Previous moves: {len(json.loads(dataset[0]['move_history']))}")

## Load Pretrained dataset

In [ ]:
print("Loading dataset...")
from datasets import load_dataset, concatenate_datasets

# Use existing dataset location
data_files = "Dataset_v5/*.json"
cache_dir = "./data_cache" 

full_dataset = load_dataset("json", data_files=data_files, split="train", cache_dir=cache_dir)
print(f"Loaded {len(full_dataset)} training examples (raw).")

# ─── Curriculum Phase 1: Small Boards (6x6, 8x8) ──────────────────────────
print("\n=== PREPARING PHASE 1: Small Boards (6x6, 8x8) ===")

def is_phase_1(example):
    return example.get("board_size") in ["6x6", "8x8"]
    
dataset_p1 = full_dataset.filter(is_phase_1)
dataset_p1 = dataset_p1.shuffle(seed=42)
print(f"Phase 1 Dataset Size: {len(dataset_p1)}")

if len(dataset_p1) == 0:
    print("WARNING: Phase 1 dataset empty! Checking board sizes...")
    print(set(full_dataset["board_size"]))
    # Fallback to full if empty
    dataset_p1 = full_dataset
    


In [ ]:
import random
dataset_2[random.randint(0,5000)]

# Configure GRPO Training

Set up GRPO trainer with all hyperparameters:

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Calculate max lengths
max_prompt_length = 5000   # JSON state prompt
max_completion_length = max_seq_length - max_prompt_length

# GRPO Configuration
training_args_p1 = GRPOConfig(
    output_dir = "minesweeper_curriculum_p1_llama",
    temperature = 1.0,
    learning_rate = 1e-5,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_prompt_length = 3500,
    max_completion_length = 128,
    max_steps = 100, # Phase 1 steps
    save_steps = 50,
    report_to = "none",
)

print("Training configuration:")
print(f"  Max steps: {training_args_p1.max_steps}")
print(f"  Generations per state: {training_args_p1.num_generations}")
print(f"  Learning rate: {training_args_p1.learning_rate}")
print(f"  LoRA rank: {lora_rank}")

In [ ]:
from transformers import TrainerCallback
class MinesweeperEvalCallback(TrainerCallback):
    def __init__(self, eval_every_steps=50, num_games=5):
        self.eval_every_steps = eval_every_steps
        self.num_games = num_games

    def on_step_end(self, args, state, control, model=None, processing_class=None, **kwargs):
        if state.global_step % self.eval_every_steps != 0:
            return

        tokenizer = processing_class
        if tokenizer is None or model is None:
            return

        was_training = model.training
        model.eval()

        wins = 0
        # Eval on Phase 1 board sizes primarily, but mix in regular
        eval_sizes = [(6, 6), (8, 8)]
        
        for i in range(self.num_games):
            e_rows, e_cols = random.choice(eval_sizes)
            e_mines = int(e_rows * e_cols * 0.20)
            # Ensure at least 1 mine
            e_mines = max(1, e_mines)
            
            game = MinesweeperGame(rows=e_rows, cols=e_cols, num_mines=e_mines, seed=20000 + i + state.global_step)
            
            # --- FIX: Play a random safe move first ---
            safe_cells = [(r, c) for r in range(game.rows) for c in range(game.cols) if game._board[r][c] != -1]
            if safe_cells:
                r_start, c_start = random.choice(safe_cells)
                game.do_action({"type": "reveal", "row": r_start, "col": c_start})
            # ----------------------------------------

            moves = 0
            while game.state() == "ongoing" and moves < 50:
                prompt = format_state_for_llm(game, hide_hints=False)
                text = tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                
                with torch.no_grad():
                    output = model.generate(
                        **tokenizer(text, return_tensors="pt").to(model.device),
                        temperature=0.7,
                        max_new_tokens=128,
                        do_sample=True,
                    )
                response = tokenizer.decode(output[0], skip_special_tokens=True)
                action = parse_llm_action(response)
                if action is None:
                    break
                game.do_action(action)
                moves += 1
            if game.state() == "success":
                wins += 1

        win_rate = wins / self.num_games
        print(f"\n[Eval @ step {state.global_step}] Win rate: {wins}/{self.num_games} ({win_rate*100:.0f}%)\n")

        if was_training:
            model.train()

eval_callback = MinesweeperEvalCallback(eval_every_steps=50, num_games=5)

# Train the Model

Start GRPO training with reward functions:

In [ ]:
trainer_p1 = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [valid_json_reward, gameplay_scores, log_completions],
    args = training_args_p1,
    train_dataset = dataset_p1,
    callbacks = [MinesweeperEvalCallback(eval_every_steps=20, num_games=3)], # More frequent eval for short phase
)
trainer_p1.train()


In [ ]:
print("Saving Phase 1 model...")
model.save_pretrained("minesweeper_grpo_p1")
tokenizer.save_pretrained("minesweeper_grpo_p1")

# ─── Curriculum Phase 2: Large Boards (10x10, 15x15, 20x20) ───────────────
print("\n=== PREPARING PHASE 2: Large Boards (10x10, 15x15, 20x20) ===")

In [ ]:
def is_phase_2(example):
    return example.get("board_size") in ["10x10", "15x15", "20x20"]
    
dataset_p2 = full_dataset.filter(is_phase_2)
dataset_p2 = dataset_p2.shuffle(seed=42)
print(f"Phase 2 Dataset Size: {len(dataset_p2)}")

if len(dataset_p2) == 0:
    print("WARNING: Phase 2 dataset empty! Using full dataset.")
    dataset_p2 = full_dataset


In [ ]:
training_args_p2 = GRPOConfig(
    output_dir = "minesweeper_curriculum_p2_llama",
    temperature = 1.0,
    learning_rate = 5e-5, # Keep same LR or decay? Keeping same for simplicity
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_prompt_length = 3500,
    max_completion_length = 512,
    max_steps = 300, # Run for another 100 steps (Total 200 essentially)
    save_steps = 50,
    report_to = "none",
)

print("Starting Phase 2 Training...")
# Re-init trainer with new dataset and args
# Model object preserves weights from Phase 1
trainer_p2 = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [valid_json_reward, gameplay_scores, log_completions],
    args = training_args_p2,
    train_dataset = dataset_p2,
    callbacks = [MinesweeperEvalCallback(eval_every_steps=20, num_games=3)],
)
trainer_p2.train()



In [ ]:
print("Saving Final Model (Phase 2)...")
model.save_pretrained("minesweeper_grpo_final")
tokenizer.save_pretrained("minesweeper_grpo_final")


# Test Trained Model

Evaluate the finetuned model:

In [ ]:
# Test on new game
test_game = MinesweeperGame(rows=8, cols=8, num_mines=20)
test_prompt = format_state_for_llm(test_game)

# Removed reasoning_effort="low" for train/eval consistency
test_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize = False,
    add_generation_prompt = True,
)

print("=== Trained Model Response ===")
output = model.generate(
    **tokenizer(test_text, return_tensors = "pt").to("cuda"),
    temperature = 0.1,
    max_new_tokens = 128,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

# Parse and test action
response_text = tokenizer.decode(output[0])
action = parse_llm_action(response_text)
print(f"\nParsed action: {action}")

if action:
    test_game.do_action(action)
    print(f"\nGame state after action: {test_game.state()}")
    print(test_game.pretty_print())

# Evaluation: Play Complete Games

Test the model on multiple complete games:

In [ ]:
def play_full_game(model, tokenizer, rows=6, cols=6, num_mines=5, seed=None, max_moves=50):
    """Play a complete Minesweeper game with the model"""
    game = MinesweeperGame(rows=rows, cols=cols, num_mines=num_mines, seed=seed)
    moves = 0
    
    while game.state() == "ongoing" and moves < max_moves:
        # Get current state
        prompt = format_state_for_llm(game)
        # Removed reasoning_effort="low" for train/eval consistency
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize = False,
            add_generation_prompt = True,
        )
        
        # Generate action
        output = model.generate(
            **tokenizer(text, return_tensors = "pt").to("cuda"),
            temperature = 0.7,
            max_new_tokens = 128,
            do_sample = True,
        )
        
        response = tokenizer.decode(output[0])
        action = parse_llm_action(response)
        
        if action is None:
            break  # Invalid action
        
        game.do_action(action)
        moves += 1
    
    return game, moves

# Evaluate on 100 games for statistically meaningful win rates
NUM_EVAL_GAMES = 100
print(f"Evaluating model on {NUM_EVAL_GAMES} games...\n")
wins = 0
total_moves = 0

for i in range(NUM_EVAL_GAMES):
    game, moves = play_full_game(model, tokenizer, seed=i)
    result = game.state()

    if result == "success":
        wins += 1
    # Only print individual results for first 10 + any wins
    if i < 10 or result == "success":
        tag = "WIN" if result == "success" else "LOSS"
        print(f"Game {i+1}: {tag} ({result}) after {moves} moves")

    total_moves += moves

if NUM_EVAL_GAMES > 10:
    print(f"... (showing first 10 + wins; {NUM_EVAL_GAMES} total)")

print(f"\nResults:")
print(f"  Win rate: {wins}/{NUM_EVAL_GAMES} ({wins/NUM_EVAL_GAMES*100:.1f}%)")
print(f"  Average moves: {total_moves/NUM_EVAL_GAMES:.1f}")

# Save the Model

Save your trained model for competition submission:

In [ ]:
# Save LoRA adapters
import os
folder= "trained_models"
os.mkdir(folder)
model.save_pretrained(f"{folder}/my_minesweeper_model_v1_llama")
tokenizer.save_pretrained(f"{folder}/my_minesweeper_v1_llama")

print("Model saved to: my_minesweeper_model/")

# Save merged model in 16bit (local file name which will be used for eval)
if False:
    model.save_pretrained_merged(
        "my_minesweeper_model_merged_LLAMA_v1",
        tokenizer,
        save_method = "merged_16bit"
    )

# Competition Tips

## Improve Your Model:

1. **Adjust Reward Functions**
   - Increase rewards for logical deduction
   - Add penalties for random moves
   - Reward flagging correct mines

2. **Tune Hyperparameters**
   - Increase `max_steps` for longer training
   - Adjust `learning_rate` (try 1e-5 to 1e-4)
   - Increase `lora_rank` for more capacity
   - Adjust `num_generations` (2-8)

3. **Better Training Data**
   - Generate more diverse states
   - Include harder scenarios (more mines)
   - Add states requiring logical deduction

4. **Advanced Techniques**
   - Multi-step rollouts in reward function
   - Curriculum learning (easy → hard boards)
   - Ensemble multiple models

## Team Strategy:
- Experiment with different reward functions
- Try different board sizes during training
- Analyze failed games to improve rewards
- Use temperature sampling during evaluation

Good luck!